# UC Food Waste · RGB+Thermal

아래 코드 셀에서 **Shift+Enter**를 누르세요. Drive 연결 요청이 뜨면 승인하세요. NPZ와 마스크가 확인되면 1 epoch를 실행하고 Test 점수를 표시합니다.


In [ ]:
"""Locate Colab data and start the RGB+Thermal trial."""

import csv
import os
from pathlib import Path
import subprocess
import sys
from zipfile import ZipFile


def locate(names, configured):
    if configured:
        candidates = [Path(configured)]
    else:
        result = subprocess.run(
            ["find", "-L", "/content", "-type", "f", "(",
             "-name", "image000010.npz", "-o", "-name", "data_processed.zip", ")"],
            capture_output=True, text=True,
        )
        candidates = [Path(line) for line in result.stdout.splitlines()]
    folders = {p.parent if p.suffix == ".npz" else p for p in candidates}
    npz = next((p for p in sorted(folders)
                if all((p / f"{name}.npz").is_file() for name in names)), None)
    return npz, [p for p in candidates if p.name == "data_processed.zip"]


def main():
    with Path("metadata/data_split.csv").open(encoding="utf-8-sig") as stream:
        names = sorted(row["filename"] for row in csv.DictReader(stream))
    configured = os.environ.get("UC_FOODWASTE_NPZ_DIR")
    print("Colab 데이터 위치 확인 중...", flush=True)
    npz, archives = locate(names, configured)
    if npz is None:
        raise SystemExit(
            "현재 Colab에서 split CSV에 대응하는 NPZ 542개가 있는 폴더를 찾지 못했습니다. "
            "NPZ가 다른 계정의 '공유 문서함'에 있으면 Google Drive에서 "
            "해당 폴더의 바로가기를 내 드라이브에 추가한 뒤 다시 실행하세요."
        )
    masks = Path("data/data_processed/masks")
    if not all((masks / f"{n}.png").is_file() for n in names):
        if configured:
            _, archives = locate([], None)
        for path in sorted(archives):
            with ZipFile(path) as archive:
                entries = {
                    Path(name).stem: name for name in archive.namelist()
                    if Path(name).parent.name == "masks" and Path(name).suffix.lower() == ".png"
                }
                if not entries.keys() >= set(names):
                    continue
                masks.mkdir(parents=True, exist_ok=True)
                for name in names:
                    target = masks / f"{name}.png"
                    if not target.is_file():
                        target.write_bytes(archive.read(entries[name]))
                break
        else:
            raise SystemExit("마스크 542개를 포함한 data_processed.zip을 현재 Colab에서 찾지 못했습니다.")
    print(f"NPZ 폴더: {npz}", flush=True)
    print(f"마스크 폴더: {masks.resolve()}", flush=True)
    subprocess.run([
        sys.executable, "-u", "archive/unet/scripts/train.py", "--mode", "rgb_thermal",
        "--epochs", "1", "--batch-size", "2", "--num-workers", "0",
        "--npz-dir", str(npz), "--mask-dir", str(masks),
    ], check=True)


from google.colab import drive
from IPython.display import display, Markdown
import torch

project = Path('/content/UC-FoodWaste')
if not torch.cuda.is_available():
    raise RuntimeError('런타임 → 런타임 유형 변경에서 GPU를 선택하세요.')

print('GPU:', torch.cuda.get_device_name(0), flush=True)
if not (project / '.git').is_dir():
    subprocess.run(['git', 'clone', 'https://github.com/jeehyung0922-lab/UC-FoodWaste.git', str(project)], check=True)
if not (project / 'archive/unet/scripts/train.py').is_file():
    raise RuntimeError('클론 폴더 안에 archive/unet/scripts/train.py가 없습니다.')
if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive')

print('학습 패키지 확인 중...', flush=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(project / 'archive/unet/requirements.txt')], check=True)
previous_dir = Path.cwd()
try:
    os.chdir(project)
    main()
finally:
    os.chdir(previous_dir)

import json
import pandas as pd
import matplotlib.pyplot as plt

results = project / 'archive/unet/runs/rgb_thermal'
summary = json.loads((results / 'test_metrics.json').read_text())
display(Markdown('### RGB+Thermal 시험 실행 결과'))
display(pd.DataFrame([summary['overall']], index=['Test']).round(4))
overall = pd.Series(summary['overall'])
ax = overall.plot.bar(color='#3178a8', rot=0, ylim=(0, 1), figsize=(6, 3))
ax.set_ylabel('Score')
ax.set_title('Test segmentation metrics')
plt.tight_layout()
plt.show()
print('결과 파일:', results)
